In [1]:
!apt-get update
!apt-get install -y fonts-noto-cjk

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,315 kB]
Get:14 http

In [6]:
# -*- coding: utf-8 -*-
"""
모델학습_v3_정상화.py

[v2 대비 수정 사항]
1. tokenizer 정의 누락 문제 해결 -> 셀 5에 명시적으로 추가
2. is_strict_finance 함수명 불일치 해결 -> is_strict_finance로 통일, 정의/호출 일치
3. FINANCE_KEYWORDS / EXCLUDE_KEYWORDS -> 실제 검증된 값으로 확정 (아래 참고)
4. 각 단계마다 검증 로그 추가 (개수, 샘플 미리보기) -> 재현성 확보
5. train/val 분할 시 인덱스 저장 -> 나중에 오분류 분석 시 원본 텍스트 역추적 가능

[확정된 필터링 기준]
- FINANCE_KEYWORDS: 실제 학습에 쓰인 15개 버전 (펀드/주식/투자/보험/보증보험 포함)
- EXCLUDE_KEYWORDS: 더 엄격한 15개 버전 (교육 + 상품/서비스 전부 포함)
  -> 이렇게 하면 v2 실제 학습보다 더 좁게 필터링되어 정상 데이터 개수가 줄어들 수 있음
  -> [3/4] 필터링 단계 로그에서 몇 개가 남는지 반드시 확인할 것

[단계별 실행 위치 가이드]
  Colab 셀 1: 라이브러리 설치
  Colab 셀 2: Drive 마운트 + 정규화 함수 정의
  Colab 셀 3: 데이터 로드 (엄격한 필터링 적용)
  Colab 셀 4: 정규화 + Train/Val 분할
  Colab 셀 5: tokenizer 로드 + 토크나이제이션 + Dataset
  Colab 셀 6: 모델 + Trainer 구성
  Colab 셀 7: 학습 실행
  Colab 셀 8: 오분류 분석 (학습 직후 바로 실행 가능)
"""

'\n모델학습_v3_정상화.py\n\n[v2 대비 수정 사항]\n1. tokenizer 정의 누락 문제 해결 -> 셀 5에 명시적으로 추가\n2. is_strict_finance 함수명 불일치 해결 -> is_strict_finance로 통일, 정의/호출 일치\n3. FINANCE_KEYWORDS / EXCLUDE_KEYWORDS -> 실제 검증된 값으로 확정 (아래 참고)\n4. 각 단계마다 검증 로그 추가 (개수, 샘플 미리보기) -> 재현성 확보\n5. train/val 분할 시 인덱스 저장 -> 나중에 오분류 분석 시 원본 텍스트 역추적 가능\n\n[확정된 필터링 기준]\n- FINANCE_KEYWORDS: 실제 학습에 쓰인 15개 버전 (펀드/주식/투자/보험/보증보험 포함)\n- EXCLUDE_KEYWORDS: 더 엄격한 15개 버전 (교육 + 상품/서비스 전부 포함)\n  -> 이렇게 하면 v2 실제 학습보다 더 좁게 필터링되어 정상 데이터 개수가 줄어들 수 있음\n  -> [3/4] 필터링 단계 로그에서 몇 개가 남는지 반드시 확인할 것\n\n[단계별 실행 위치 가이드]\n  Colab 셀 1: 라이브러리 설치\n  Colab 셀 2: Drive 마운트 + 정규화 함수 정의\n  Colab 셀 3: 데이터 로드 (엄격한 필터링 적용)\n  Colab 셀 4: 정규화 + Train/Val 분할\n  Colab 셀 5: tokenizer 로드 + 토크나이제이션 + Dataset\n  Colab 셀 6: 모델 + Trainer 구성\n  Colab 셀 7: 학습 실행\n  Colab 셀 8: 오분류 분석 (학습 직후 바로 실행 가능)\n'

In [2]:
# 셀1 라이브러리 설치
import subprocess
import sys

print("필요한 라이브러리 설치 중...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "peft"])
print("✅ 설치 완료\n")

필요한 라이브러리 설치 중...
✅ 설치 완료



In [7]:
# ============================================================================
# [셀 2] Drive 마운트 + 정규화 함수 정의
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import random
import zipfile
import time
import shutil
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    ElectraTokenizer, ElectraForSequenceClassification,
    TrainingArguments, Trainer
)

print("✅ Drive 마운트 완료\n")

# ==================== 정규화 함수 정의 ====================

_AIHUB_FILLER_WHITELIST = ['아', '어', '음', '그', '뭐', '저', '막', '좀']
_AIHUB_FILLER_PATTERN = re.compile(
    r'\b(?:[a-z]|' + '|'.join(_AIHUB_FILLER_WHITELIST) + r')/'
)


def merge_consecutive_dual_notation(text: str) -> str:
    pattern = r'((?:\(\d+\)/\([^)]*\)\s*에?\s*){2,}\(\d+\)/\([^)]*\))'
    def merge_group(match):
        numbers = re.findall(r'\((\d+)\)/\([^)]*\)', match.group(1))
        return ''.join(numbers)
    return re.sub(pattern, merge_group, text)


def normalize_dual_notation(text: str) -> str:
    pattern = r'\((\d[\w]*)\)/\([^)]*\)'
    return re.sub(pattern, r'\1', text)


def merge_spaced_digits(text: str) -> str:
    pattern = r'(?:\d\s){4,}\d'
    return re.sub(pattern, lambda m: m.group(0).replace(' ', ''), text)


def remove_correction_marker(text: str) -> str:
    return text.replace('+', ' ')


def remove_noise_markup(text: str) -> str:
    pattern = r'\(\([^)]*\)\)'
    return re.sub(pattern, ' ', text)


def remove_aihub_filler_markup(text: str) -> str:
    return _AIHUB_FILLER_PATTERN.sub(' ', text)


def remove_masking_marker(text: str) -> str:
    pattern = r'\(삐-?\)'
    return re.sub(pattern, ' ', text)


def strip_speaker_label(text: str) -> str:
    text = re.sub(r'(피해자|사기범)\s*:\s*', ' ', text)
    return text


def clean_whitespace(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_finance_text(text: str) -> str:
    """AI Hub + his_voice 통합 정규화"""
    text = merge_consecutive_dual_notation(text)
    text = normalize_dual_notation(text)
    text = merge_spaced_digits(text)
    text = remove_correction_marker(text)
    text = remove_noise_markup(text)
    text = remove_aihub_filler_markup(text)
    text = remove_masking_marker(text)
    text = strip_speaker_label(text)
    text = clean_whitespace(text)
    return text


print("✅ 정규화 함수 정의 완료\n")

Mounted at /content/drive
✅ Drive 마운트 완료

✅ 정규화 함수 정의 완료



In [16]:
# ============================================================================
# [셀 3] 데이터 로드 (엄격한 도메인 필터링 + 금융무관 데이터 고정 확보)
# ============================================================================

def load_txt_files(folder_path, label):
    data = []
    files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    for fname in files:
        path = os.path.join(folder_path, fname)
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read()
        data.append({'text': text, 'label': label, 'source': fname})
    return data


print("=" * 70)
print("데이터 로드 중...")
print("=" * 70 + "\n")

start_time = time.time()

# [1] 사기 데이터 (his_voice)
print("[1/4] his_voice (사기 데이터) 로드...")
his_voice_path = "/content/drive/MyDrive/his_voice"
data_path = os.path.join(his_voice_path, "data")

fraud_data = load_txt_files(os.path.join(data_path, "impersonation"), label=1)
fraud_data += load_txt_files(os.path.join(data_path, "loanScam"), label=1)
print(f"✅ 사기 데이터: {len(fraud_data)}개\n")

# [2] 정상 데이터 (AI Hub zip)
print("[2/4] AI Hub (정상 데이터) zip 압축해제...")
aihub_zip_path = "/content/drive/MyDrive/voice_pishing_text_merged/상담음성_재병합_결과.zip"
extract_path = "/tmp/aihub_extract_new"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)
os.makedirs(extract_path)

with zipfile.ZipFile(aihub_zip_path, 'r') as zf:
    zf.extractall(extract_path)

normal_data = []
files = [f for f in os.listdir(extract_path) if f.endswith('.txt')]
print(f"  추출된 파일 확인: {len(files)}개")

for fname in files:
    path = os.path.join(extract_path, fname)
    try:
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read()
        normal_data.append({'text': text, 'label': 0, 'source': fname})
    except Exception as e:
        print(f"  경고: {fname} 읽기 실패 - {e}")

print(f"✅ 정상 데이터: {len(normal_data)}개 (압축해제 후)\n")

# 검증: 이 개수가 49,839와 일치하는지 확인 (데이터 손실 재발 여부 체크)
if len(normal_data) != 49839:
    print(f"⚠️ 경고: 예상 개수(49,839)와 다름 -> 데이터 손실 재발 가능성, 원인 재확인 필요\n")
else:
    print(f"✅ 예상 개수(49,839)와 일치 -> 데이터 손실 문제 없음 확인됨\n")

# [3] 엄격한 도메인 필터링
print("[3/4] 도메인 필터링 (금융거래만, 교육/상품/서비스 제외 - 엄격 버전)...")

# 확정: 실제 학습 버전 (펀드/주식/투자/보험/보증보험 포함, 15개)
FINANCE_KEYWORDS_STRICT = [
    '은행', '계좌', '통장', '이체', '송금', '대출', '신용카드', '체크카드',
    '적금', '예금', '펀드', '주식', '투자', '보험', '보증보험'
]

# 확정: 더 엄격한 버전 (교육 + 상품/서비스 전부, 15개)
EXCLUDE_KEYWORDS = [
    '교육', '학습', '강의', '상담사',
    '클러치', '그래픽카드', '자동차',
    '택배', '배송', '구매', '상품',
    '게임', '영화', '클럽', '카페'
]


def is_strict_finance(text):
    """엄격한 금융 필터: 금융키워드 포함 AND 교육/상품/서비스키워드 제외"""
    has_finance = any(kw in text for kw in FINANCE_KEYWORDS_STRICT)
    has_exclude = any(kw in text for kw in EXCLUDE_KEYWORDS)
    return has_finance and not has_exclude


# 필터링 적용
finance_normal = [d for d in normal_data if is_strict_finance(d['text'])]
print(f"✅ 필터링 후 금융 상담: {len(finance_normal)}개 (원본 {len(normal_data)}개 중)")
print(f"   필터링 비율: {len(finance_normal) / len(normal_data):.2%}\n")

# 검증: 필터링 결과가 너무 적으면 경고 (Hard Negative 테스트에 쓸 수 있는 양인지)
if len(finance_normal) < len(fraud_data):
    print(f"⚠️ 경고: 필터링된 정상 데이터({len(finance_normal)}개)가 사기 데이터({len(fraud_data)}개)보다 적음")
    print(f"   -> EXCLUDE_KEYWORDS가 너무 엄격했을 가능성. 클래스 균형 조정 시 사기 데이터 쪽이 줄어듦\n")

# ============================================================================
# ★ [신규] 금융 무관 정상 데이터 고정 개수 확보 (팀장님 지시 반영)
# 비율 유지 방식은 폐기 -> 무조건 N_EXTRA개는 학습에 포함되도록 보장
# ============================================================================
non_finance_normal = [d for d in normal_data if not is_strict_finance(d['text'])]
print(f"✅ 금융 무관 정상 데이터 풀: {len(non_finance_normal)}개 (원본 {len(normal_data)}개 중)")

# ★ 여기 숫자만 바꿔가며 실험 (40 -> 80 -> 120 등)
N_EXTRA = 150

random.seed(42)

if len(non_finance_normal) < N_EXTRA:
    print(f"⚠️ 경고: 풀이 N_EXTRA({N_EXTRA})보다 작아서 전체({len(non_finance_normal)}개)만 사용")
    N_EXTRA = len(non_finance_normal)

extra_normal = random.sample(non_finance_normal, N_EXTRA)
for d in extra_normal:
    d['source'] = f"[non_finance]_{d['source']}"

# [4] 클래스 균형 조정 (금융무관 N_EXTRA개는 고정 확보, 나머지를 금융도메인으로 채움)
print("[4/4] 클래스 균형 조정 (금융무관 데이터 고정 확보)...")
random.seed(42)

n_before_fraud = len(fraud_data)
n_before_finance_normal = len(finance_normal)

target_n = len(fraud_data)  # 사기 데이터 개수에 정상 데이터 총합을 맞춘다

if target_n <= N_EXTRA:
    print(f"⚠️ 경고: 사기 데이터({target_n}개)가 N_EXTRA({N_EXTRA}개)보다 적거나 같음. N_EXTRA 조정 필요")
    finance_normal_final = []
    extra_normal_final = random.sample(extra_normal, target_n)
else:
    n_finance_needed = target_n - N_EXTRA
    if len(finance_normal) >= n_finance_needed:
        finance_normal_final = random.sample(finance_normal, n_finance_needed)
    else:
        print(f"⚠️ 경고: 금융도메인 정상 데이터가 부족함 ({len(finance_normal)}개 < 필요 {n_finance_needed}개)")
        finance_normal_final = finance_normal
    extra_normal_final = extra_normal

# 최종 정상 데이터 = 금융 정상 + 금융무관 정상
finance_normal = finance_normal_final + extra_normal_final

# ★ Hard Negative 42개 추가
finance_normal = finance_normal + hard_negative_list

print(f"✅ 최종 데이터:")
print(f"   - 사기(1): {len(fraud_data)}개 (조정 전 {n_before_fraud}개)")
print(f"   - 정상(0) 중 금융 도메인: {len(finance_normal_final)}개 (조정 전 {n_before_finance_normal}개)")
print(f"   - 정상(0) 중 금융 무관: {len(extra_normal_final)}개 (확보 목표 {N_EXTRA}개)")
print(f"   - 정상(0) 합계: {len(finance_normal)}개")
print(f"   - 전체 합계: {len(fraud_data) + len(finance_normal)}개\n")

elapsed = time.time() - start_time
print(f"📊 데이터 로드 완료 ({elapsed:.1f}초)\n\n")

데이터 로드 중...

[1/4] his_voice (사기 데이터) 로드...
✅ 사기 데이터: 239개

[2/4] AI Hub (정상 데이터) zip 압축해제...
  추출된 파일 확인: 49839개
✅ 정상 데이터: 49839개 (압축해제 후)

✅ 예상 개수(49,839)와 일치 -> 데이터 손실 문제 없음 확인됨

[3/4] 도메인 필터링 (금융거래만, 교육/상품/서비스 제외 - 엄격 버전)...
✅ 필터링 후 금융 상담: 5126개 (원본 49839개 중)
   필터링 비율: 10.29%

✅ 금융 무관 정상 데이터 풀: 44713개 (원본 49839개 중)
[4/4] 클래스 균형 조정 (금융무관 데이터 고정 확보)...
✅ 최종 데이터:
   - 사기(1): 239개 (조정 전 239개)
   - 정상(0) 중 금융 도메인: 89개 (조정 전 5126개)
   - 정상(0) 중 금융 무관: 150개 (확보 목표 150개)
   - 정상(0) 합계: 284개
   - 전체 합계: 523개

📊 데이터 로드 완료 (20.7초)




In [17]:
# ============================================================================
# [셀 4] 정규화 + Train/Val 분할
# ============================================================================

print("=" * 70)
print("정규화 및 학습 데이터 준비")
print("=" * 70 + "\n")

start_time = time.time()

# 정규화
all_data = fraud_data + finance_normal
print(f"정규화 적용 중 ({len(all_data)}개)...")
for i, item in enumerate(all_data):
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(all_data)}")
    item['text'] = normalize_finance_text(item['text'])
print("✅ 정규화 완료\n")

# 검증: 정규화 후 빈 텍스트가 생겼는지 확인 (정규화 과정에서 내용이 다 지워지는 엣지 케이스 방지)
empty_after_norm = sum(1 for item in all_data if len(item['text'].strip()) == 0)
if empty_after_norm > 0:
    print(f"⚠️ 경고: 정규화 후 빈 텍스트 {empty_after_norm}개 발견 -> 해당 샘플 확인 필요\n")

# 데이터프레임 + Train/Val 분할 (원본 인덱스 보존 -> 나중에 역추적용)
df = pd.DataFrame(all_data).reset_index(drop=True)
df['orig_idx'] = df.index  # 오분류 분석 시 원본 source 파일명 역추적용

print("라벨 분포:")
print(df['label'].value_counts())
print()

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"✅ Train: {len(train_df)}, Val: {len(val_df)}")
print(f"   Train 라벨 분포: {dict(train_df['label'].value_counts())}")
print(f"   Val 라벨 분포  : {dict(val_df['label'].value_counts())}\n")

elapsed = time.time() - start_time
print(f"📊 준비 완료 ({elapsed:.1f}초)\n\n")

정규화 및 학습 데이터 준비

정규화 적용 중 (523개)...
  100/523
  200/523
  300/523
  400/523
  500/523
✅ 정규화 완료

라벨 분포:
label
0    284
1    239
Name: count, dtype: int64

✅ Train: 418, Val: 105
   Train 라벨 분포: {0: np.int64(227), 1: np.int64(191)}
   Val 라벨 분포  : {0: np.int64(57), 1: np.int64(48)}

📊 준비 완료 (0.2초)




In [21]:
# ============================================================================
# [셀 5] tokenizer 로드 + 청킹 + 토크나이제이션 + Dataset
# ============================================================================
# ★ v3-chunked 수정: 128토큰 초과 텍스트를 앞부분만 자르지 않고
#   128토큰 단위로 순차 분할(overlap 없음)하여 뒷부분 정보 유실을 방지한다.
#   각 청크는 원본의 orig_idx / source / label을 그대로 상속받는다.
#   (추론 시 원본 판정 = 청크 중 하나라도 사기(1)면 사기로 판정 — OR 로직,
#    셀 8-오분류 분석에서 그룹핑하여 적용)

print("=" * 70)
print("Tokenizer 로드 및 청킹")
print("=" * 70 + "\n")

start_time = time.time()

MODEL_NAME = "skplanet/dialog-koelectra-small-discriminator"

print(f"Tokenizer 로드: {MODEL_NAME}")
tokenizer = ElectraTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer 로드 완료 (vocab size: {tokenizer.vocab_size})\n")

MAX_LENGTH = 128


def chunk_text_by_tokens(text, tokenizer, max_length=128):
    """
    텍스트를 max_length 토큰 단위로 순차 분할한다 (overlap 없음).
    special token(CLS/SEP) 자리를 남겨두기 위해 실제 콘텐츠 토큰 길이는
    max_length - 2로 잡고, 각 청크를 다시 텍스트로 decode해서 반환한다.
    (Dataset 단계에서 tokenizer가 CLS/SEP을 다시 붙이므로 여기서는 순수 콘텐츠만 자른다)
    """
    content_len = max_length - 2  # [CLS], [SEP] 자리 확보
    token_ids = tokenizer.encode(text, add_special_tokens=False)

    if len(token_ids) <= content_len:
        return [text]  # 짧으면 청킹 불필요, 원본 그대로 1개 청크

    chunks = []
    for start in range(0, len(token_ids), content_len):
        chunk_ids = token_ids[start:start + content_len]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        if chunk_text.strip():  # 빈 청크 방지
            chunks.append(chunk_text)
    return chunks


def build_chunked_df(df, tokenizer, max_length=128):
    """
    원본 df(한 행 = 원본 상담 1건)를 청크 단위 df로 확장한다.
    반환되는 df는 원본보다 행 수가 많아지며, orig_idx로 원본 그룹핑이 가능하다.
    """
    rows = []
    n_multi_chunk = 0  # 청킹이 실제로 일어난(2개 이상 청크) 원본 개수 카운트
    for _, row in df.iterrows():
        chunks = chunk_text_by_tokens(row['text'], tokenizer, max_length=max_length)
        if len(chunks) > 1:
            n_multi_chunk += 1
        for chunk_idx, chunk_text in enumerate(chunks):
            rows.append({
                'text': chunk_text,
                'label': row['label'],
                'source': row['source'],
                'orig_idx': row['orig_idx'],
                'chunk_idx': chunk_idx,
                'n_chunks': len(chunks),
            })
    chunked_df = pd.DataFrame(rows).reset_index(drop=True)
    print(f"  원본 {len(df)}개 -> 청크 {len(chunked_df)}개 (128토큰 초과로 분할된 원본: {n_multi_chunk}개)")
    return chunked_df


print("Train 청킹 중...")
train_df_chunked = build_chunked_df(train_df, tokenizer, max_length=MAX_LENGTH)
print("Val 청킹 중...")
val_df_chunked = build_chunked_df(val_df, tokenizer, max_length=MAX_LENGTH)
print()

# 검증: 청킹 후에도 라벨 분포가 원본과 비슷한 비율을 유지하는지 확인
print("청킹 후 라벨 분포:")
print(f"  Train: {dict(train_df_chunked['label'].value_counts())}")
print(f"  Val  : {dict(val_df_chunked['label'].value_counts())}\n")

# 검증: orig_idx가 val_df와 train_df에서 서로 겹치지 않는지 (data leakage 방지 확인)
overlap = set(train_df_chunked['orig_idx']) & set(val_df_chunked['orig_idx'])
if len(overlap) > 0:
    print(f"⚠️ 경고: Train/Val 간 orig_idx 겹침 {len(overlap)}개 발견 -> 데이터 누수 가능성\n")
else:
    print("✅ Train/Val 간 orig_idx 겹침 없음 확인\n")

train_encodings = tokenizer(
    train_df_chunked['text'].tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH
)
val_encodings = tokenizer(
    val_df_chunked['text'].tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH
)

print(f"✅ 청크 토크나이제이션 완료 (max_length={MAX_LENGTH})\n")


# Dataset 클래스
class VoicePhishingDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item


train_dataset = VoicePhishingDataset(train_encodings, train_df_chunked['label'].tolist())
val_dataset = VoicePhishingDataset(val_encodings, val_df_chunked['label'].tolist())

print("✅ Dataset 구성 완료 (청크 기준)\n")

elapsed = time.time() - start_time
print(f"📊 청킹 + 토크나이제이션 완료 ({elapsed:.1f}초)\n\n")


Tokenizer 로드 및 청킹

Tokenizer 로드: skplanet/dialog-koelectra-small-discriminator


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (472 > 128). Running this sequence through the model will result in indexing errors


✅ Tokenizer 로드 완료 (vocab size: 39999)

Train 청킹 중...
  원본 418개 -> 청크 3112개 (128토큰 초과로 분할된 원본: 340개)
Val 청킹 중...
  원본 105개 -> 청크 906개 (128토큰 초과로 분할된 원본: 81개)

청킹 후 라벨 분포:
  Train: {1: np.int64(1653), 0: np.int64(1459)}
  Val  : {1: np.int64(565), 0: np.int64(341)}

✅ Train/Val 간 orig_idx 겹침 없음 확인

✅ 청크 토크나이제이션 완료 (max_length=128)

✅ Dataset 구성 완료 (청크 기준)

📊 청킹 + 토크나이제이션 완료 (3.8초)




In [24]:
# ============================================================================
# [셀 6] 모델 + Trainer 구성
# ============================================================================

print("=" * 70)
print("모델 로드 및 Trainer 구성")
print("=" * 70 + "\n")

print("모델 로드...")
model = ElectraForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
print("✅ 모델 로드 완료\n")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


print("Trainer 구성...")
OUTPUT_DIR = "/content/drive/MyDrive/results_v7_domain_expanded_150"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

os.environ["WANDB_DISABLED"] = "true"

print("✅ Trainer 구성 완료\n\n")

모델 로드 및 Trainer 구성

모델 로드...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: skplanet/dialog-koelectra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on yo

✅ 모델 로드 완료

Trainer 구성...
✅ Trainer 구성 완료




In [25]:
# ============================================================================
# [셀 7] 학습 실행
# ============================================================================

print("=" * 70)
print("🚀 학습 시작")
print("=" * 70 + "\n")

trainer.train()

print("\n" + "=" * 70)
print("✅ 학습 완료")
print("=" * 70)
print(f"\n📁 모델 저장 위치: {OUTPUT_DIR}")

# 학습 설정도 함께 저장 -> 재현성 확보 (나중에 "이 모델이 어떤 필터링으로 나왔는지" 추적 가능)
import json
run_metadata = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "finance_keywords": FINANCE_KEYWORDS_STRICT,
    "exclude_keywords": EXCLUDE_KEYWORDS,
    "n_fraud": len(fraud_data),
    "n_normal": len(finance_normal),
    "n_train": len(train_df),
    "n_val": len(val_df),
    "aihub_raw_count": len(normal_data),
}
with open(os.path.join(OUTPUT_DIR, "run_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, ensure_ascii=False, indent=2)
print(f"📄 학습 메타데이터 저장: {os.path.join(OUTPUT_DIR, 'run_metadata.json')}\n")

🚀 학습 시작



Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.136603,0.049715,0.988962,0.992895,0.989381,0.991135
2,0.004491,0.050430,0.990066,0.992908,0.991150,0.992028
3,0.003682,0.045071,0.990066,0.992908,0.991150,0.992028


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ 학습 완료

📁 모델 저장 위치: /content/drive/MyDrive/results_v7_domain_expanded_150
📄 학습 메타데이터 저장: /content/drive/MyDrive/results_v7_domain_expanded_150/run_metadata.json



In [14]:
import torch
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

# CSV 로드
df = pd.read_csv("/content/drive/MyDrive/e2e_evaluation_results_full.csv")

# v6 모델로 재예측
model = trainer.model
model.eval()
device = next(model.parameters()).device

def predict_with_chunking(text, model, tokenizer, max_length=128):
    chunks = chunk_text_by_tokens(text, tokenizer, max_length=max_length)
    if len(chunks) == 0:
        return 0
    for chunk in chunks:
        inputs = tokenizer(
            chunk, return_tensors="pt", truncation=True,
            max_length=max_length, padding=True
        ).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            pred = torch.argmax(logits, dim=-1).item()
        if pred == 1:
            return 1
    return 0

print("v6 모델로 재예측 중...")
new_preds = []
for i, row in df.iterrows():
    pred = predict_with_chunking(row['stt_text'], model, tokenizer)
    new_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(df)} 완료")

df['pred_label_v6'] = new_preds

# 비교
old_precision = precision_score(df['true_label'], df['pred_label'])
new_precision = precision_score(df['true_label'], df['pred_label_v6'])

print("\n" + "=" * 60)
print("📊 v6 재검증 결과")
print("=" * 60)
print(f"Precision: {old_precision:.4f} → {new_precision:.4f}")
print("=" * 60)

v6 모델로 재예측 중...
  10/100 완료
  20/100 완료
  30/100 완료
  40/100 완료
  50/100 완료
  60/100 완료
  70/100 완료
  80/100 완료
  90/100 완료
  100/100 완료

📊 v6 재검증 결과
Precision: 0.5158 → 0.5213


In [15]:
# ============================================================================
# [셀 8] Hard Negative 42개 추출 (v7용 데이터 준비)
# ============================================================================

# 42개 오탐 추출
still_wrong_v6 = df[(df['true_label'] == 0) & (df['pred_label_v6'] == 1)]

hard_negative_list = []
for text in still_wrong_v6['stt_text'].values:
    hard_negative_list.append({
        'text': text,
        'label': 0,
        'source': '[hard_negative]'
    })

print(f"✅ Hard Negative 42개 준비 완료")
print(f"샘플: {hard_negative_list[0]['text'][:80]}...\n")

✅ Hard Negative 42개 준비 완료
샘플: 아 네. 아니요. 그 다름이 아니라 제가 얼마 전에 그 서울시에 안 같이 하고 있는 그 인사 담당자 통해서 검사를 했는데 결과 결과 계속 해주면...



In [ ]:
# ============================================================================
# [셀 8] 오분류 분석 (청크 단위 예측 -> 원본 단위 OR 로직 집계)
# ============================================================================
# ★ v3-chunked 수정: val_dataset은 청크 기준(96개 -> 약 N개)이므로
#   기존처럼 val_df와 1:1로 매핑할 수 없다.
#   대신 val_df_chunked의 orig_idx로 그룹핑하여 원본 단위 판정을 복원한다.
#   원본 판정 로직(OR): 그룹 내 청크 중 하나라도 pred_label==1이면 원본을 사기(1)로 판정

print("=" * 70)
print("오분류 샘플 분석 (청크 단위 -> 원본 단위 OR 집계)")
print("=" * 70 + "\n")

eval_ds = trainer.eval_dataset

print(f"Validation 청크 전체 개수: {len(eval_ds)}")
labels_arr = np.array(eval_ds.labels)
unique, counts = np.unique(labels_arr, return_counts=True)
print("청크 라벨별 개수:")
for u, c in zip(unique, counts):
    tag = "사기(1)" if u == 1 else "정상(0)"
    print(f"  {tag}: {c}개")
print()

pred_output = trainer.predict(eval_ds)
logits = pred_output.predictions
probs = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
pred_labels = np.argmax(logits, axis=-1)
true_labels = pred_output.label_ids
confidences = probs.max(axis=-1)

texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in eval_ds.encodings["input_ids"]]

# 청크 단위 결과 (val_df_chunked와 순서가 동일함 -> 그대로 컬럼 결합)
chunk_result_df = pd.DataFrame({
    "orig_idx": val_df_chunked["orig_idx"].tolist(),
    "source": val_df_chunked["source"].tolist(),
    "chunk_idx": val_df_chunked["chunk_idx"].tolist(),
    "n_chunks": val_df_chunked["n_chunks"].tolist(),
    "text": texts,
    "true_label": true_labels,
    "pred_label": pred_labels,
    "confidence": confidences,
})
chunk_result_df["correct"] = chunk_result_df["true_label"] == chunk_result_df["pred_label"]

print(f"[청크 단위] 총 오분류: {(~chunk_result_df['correct']).sum()}개 / 전체 {len(chunk_result_df)}개\n")

# ---------------------------------------------------------------
# 원본 단위 집계 (OR 로직: 청크 중 하나라도 사기 예측이면 원본을 사기로 판정)
# ---------------------------------------------------------------
def aggregate_or(group):
    return pd.Series({
        "true_label": group["true_label"].iloc[0],  # 원본 라벨은 모든 청크가 동일
        "pred_label": 1 if (group["pred_label"] == 1).any() else 0,
        "n_chunks": group["n_chunks"].iloc[0],
        "max_confidence": group["confidence"].max(),
        "source": group["source"].iloc[0],
    })

orig_result_df = chunk_result_df.groupby("orig_idx").apply(aggregate_or).reset_index()
orig_result_df["correct"] = orig_result_df["true_label"] == orig_result_df["pred_label"]

print(f"[원본 단위 - OR 로직] 총 오분류: {(~orig_result_df['correct']).sum()}개 / 전체 {len(orig_result_df)}개\n")

wrong_orig = orig_result_df[~orig_result_df["correct"]].sort_values("max_confidence", ascending=False)
for _, row in wrong_orig.iterrows():
    true_tag = "사기" if row["true_label"] == 1 else "정상"
    pred_tag = "사기" if row["pred_label"] == 1 else "정상"
    print(f"[실제: {true_tag} / 예측: {pred_tag} / 최대확신도: {row['max_confidence']:.4f} / 청크수: {row['n_chunks']} / 원본파일: {row['source']}]")

# ---------------------------------------------------------------
# 원본 단위 Confusion Matrix 지표 (팀 보고용 핵심 숫자)
# ---------------------------------------------------------------
tn = ((orig_result_df["true_label"] == 0) & (orig_result_df["pred_label"] == 0)).sum()
fp = ((orig_result_df["true_label"] == 0) & (orig_result_df["pred_label"] == 1)).sum()
fn = ((orig_result_df["true_label"] == 1) & (orig_result_df["pred_label"] == 0)).sum()
tp = ((orig_result_df["true_label"] == 1) & (orig_result_df["pred_label"] == 1)).sum()

acc = (tp + tn) / (tp + tn + fp + fn) if (tp+tn+fp+fn) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n[원본 단위 - OR 로직] 성능:")
print(f"  TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"  Accuracy={acc:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}\n")

correct_conf = orig_result_df[orig_result_df["correct"]]["max_confidence"]
wrong_conf = orig_result_df[~orig_result_df["correct"]]["max_confidence"]

print(f"맞은 원본 확신도: 평균={correct_conf.mean():.4f}, 표준편차={correct_conf.std():.4f}, 최솟값={correct_conf.min():.4f}")
if len(wrong_conf) > 0:
    print(f"틀린 원본 확신도: 평균={wrong_conf.mean():.4f}, 표준편차={wrong_conf.std():.4f}")

print("""
※ 판단 기준:
  - 맞은 샘플 확신도 표준편차가 매우 작고(<0.01) 대부분 0.99+ 근처에 몰려있으면
    -> 형식적 단서로 구분하고 있을 가능성 (누수 재의심)
  - 표준편차가 0.02~0.1대이고 최솟값이 0.6~0.8대까지 내려가면
    -> 실제로 헷갈리는 경계 케이스 존재 -> 정상적인 학습 신호
""")

# eval_predictions.csv는 셀 다음 단계(Confusion Matrix 셀)에서 원본 단위 기준으로 읽으므로
# 컬럼명을 기존 포맷(true_label, pred_label)에 맞춰 저장한다.
orig_result_df.to_csv(os.path.join(OUTPUT_DIR, "eval_predictions.csv"), index=False, encoding="utf-8-sig")
chunk_result_df.to_csv(os.path.join(OUTPUT_DIR, "eval_predictions_chunk_level.csv"), index=False, encoding="utf-8-sig")
print(f"원본 단위 예측 결과 저장: {os.path.join(OUTPUT_DIR, 'eval_predictions.csv')}")
print(f"청크 단위 예측 결과 저장: {os.path.join(OUTPUT_DIR, 'eval_predictions_chunk_level.csv')}")


오분류 샘플 분석 (청크 단위 -> 원본 단위 OR 집계)

Validation 청크 전체 개수: 861
청크 라벨별 개수:
  정상(0): 347개
  사기(1): 514개



[청크 단위] 총 오분류: 13개 / 전체 861개

[원본 단위 - OR 로직] 총 오분류: 6개 / 전체 96개

[실제: 정상 / 예측: 사기 / 최대확신도: 0.9903 / 청크수: 7 / 원본파일: D61_J92_S00010845.txt]
[실제: 정상 / 예측: 사기 / 최대확신도: 0.9903 / 청크수: 7 / 원본파일: D61_J92_S00011691.txt]
[실제: 정상 / 예측: 사기 / 최대확신도: 0.9903 / 청크수: 8 / 원본파일: D61_J92_S00007636.txt]
[실제: 정상 / 예측: 사기 / 최대확신도: 0.9902 / 청크수: 10 / 원본파일: [non_finance]_D60_J91_S00000462.txt]
[실제: 사기 / 예측: 정상 / 최대확신도: 0.9449 / 청크수: 1 / 원본파일: 36527_제일저축은행을 사칭하면서 방문 상담을 하지 않도록 유도.txt]
[실제: 사기 / 예측: 정상 / 최대확신도: 0.6492 / 청크수: 1 / 원본파일: 36530_SC저축은행 햇살론 안내센터라며 대출 권유.txt]

[원본 단위 - OR 로직] 성능:
  TN=44, FP=4, FN=2, TP=46
  Accuracy=0.9375, Precision=0.9200, Recall=0.9583, F1=0.9388

맞은 원본 확신도: 평균=0.9909, 표준편차=0.0010, 최솟값=0.9873
틀린 원본 확신도: 평균=0.9259, 표준편차=0.1367

※ 판단 기준:
  - 맞은 샘플 확신도 표준편차가 매우 작고(<0.01) 대부분 0.99+ 근처에 몰려있으면
    -> 형식적 단서로 구분하고 있을 가능성 (누수 재의심)
  - 표준편차가 0.02~0.1대이고 최솟값이 0.6~0.8대까지 내려가면
    -> 실제로 헷갈리는 경계 케이스 존재 -> 정상적인 학습 신호

원본 단위 예측 결과

/tmp/ipykernel_6925/2182541442.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  orig_result_df = chunk_result_df.groupby("orig_idx").apply(aggregate_or).reset_index()


In [ ]:
# -*- coding: utf-8 -*-
"""
hard_negative_test.py

목적: 학습된 모델이 "긴급성/권유/개인정보 요구" 같은 사기 유사 표현이 섞인
      정상 금융 상담에서도 오탐(False Positive) 없이 판단하는지 검증

전제:
  - 모델학습_v3_정상화.py 를 이미 실행해서 trainer, tokenizer, normal_data,
    finance_normal 등이 메모리에 남아있는 상태
  - 만약 세션이 끊겼다면 셀 0부터 실행

[단계]
  셀 1: (세션 끊긴 경우만) 모델/데이터 재로드
  셀 2: Hard Negative 후보 자동 추출 (키워드 기반)
  셀 3: 예측 실행 (학습 없이 predict만)
  셀 4: 결과 분석 - 오탐(False Positive) 확인
"""

import os
import re
import numpy as np
import pandas as pd
import torch

# ============================================================================
# [셀 1] (세션이 끊긴 경우만) 모델/데이터 재로드 - 필요시 주석 해제
# ============================================================================
# from transformers import ElectraTokenizer, ElectraForSequenceClassification, Trainer, TrainingArguments
#
# MODEL_PATH = "/content/drive/MyDrive/results_v3_normalized"
# tokenizer = ElectraTokenizer.from_pretrained(MODEL_PATH)
# model = ElectraForSequenceClassification.from_pretrained(MODEL_PATH)
#
# training_args = TrainingArguments(output_dir="/tmp/hard_neg_eval", per_device_eval_batch_size=16)
# trainer = Trainer(model=model, args=training_args, tokenizer=tokenizer)
#
# # finance_normal, normalize_finance_text 도 다시 정의/로드 필요
# # -> 모델학습_v3_정상화.py의 [셀 2] 정규화 함수 정의 부분과
# #    [셀 3] 데이터 로드 부분을 먼저 재실행해야 함

# ============================================================================
# [셀 2] Hard Negative 후보 자동 추출
# ============================================================================

print("=" * 70)
print("Hard Negative 후보 추출 (정상 데이터 중 사기 유사 표현 포함)")
print("=" * 70 + "\n")

# 보이스피싱에서 자주 쓰이는 표현 카테고리별 키워드
# ※ 참고용 키워드 세트 - 실제 보이스피싱 수법 연구자료 기반이 아닌 일반적으로
#   알려진 패턴을 정리한 것이므로, 실제 사용 전 검토 권장
URGENCY_KEYWORDS = ['즉시', '지금 바로', '오늘 안에', '긴급', '빨리', '시간이 없']
AUTHORITY_KEYWORDS = ['금융감독원', '검찰', '경찰', '수사', '법원', '공문', '금감원']
INFO_REQUEST_KEYWORDS = ['인증번호', '비밀번호', '주민등록번호', '보안카드', 'OTP', '계좌번호를 알려']
MONEY_ACTION_KEYWORDS = ['이체해', '송금해', '입금해', '보내주세요', '전달해']

HARD_NEG_KEYWORDS = (
    URGENCY_KEYWORDS + AUTHORITY_KEYWORDS +
    INFO_REQUEST_KEYWORDS + MONEY_ACTION_KEYWORDS
)


def matches_hard_negative(text):
    """어떤 카테고리 키워드가 매칭됐는지도 함께 반환"""
    matched = [kw for kw in HARD_NEG_KEYWORDS if kw in text]
    return matched


# 주의: 학습 스크립트의 클래스 균형 조정 단계에서
#   finance_normal = random.sample(finance_normal, len(fraud_data))
# 로 원본 변수가 덮어써졌기 때문에, 지금 finance_normal은 5,126개가 아니라
# 학습에 실제로 쓰인 239개만 남아있는 상태.
#
# Hard Negative 테스트는 "학습에 안 쓰인 데이터"에서 뽑아야 의미가 있으므로,
# normal_data(AI Hub 원본 전체, 49,839개)에서 다시 필터링하고,
# 학습에 쓰인 239개(train_df/val_df에 포함된 source)는 제외한다.

# 학습/검증에 이미 쓰인 source 파일명 집합 (train_df, val_df가 메모리에 남아있어야 함)
used_sources = set(train_df['source'].tolist()) | set(val_df['source'].tolist())
print(f"학습+검증에 이미 쓰인 데이터: {len(used_sources)}개 (제외 대상)")

# normal_data(49,839개 원본)에 대해 다시 도메인 필터링 적용
# (is_strict_finance, FINANCE_KEYWORDS_STRICT, EXCLUDE_KEYWORDS가 메모리에 남아있어야 함)
finance_normal_full = [d for d in normal_data if is_strict_finance(d['text'])]
print(f"전체 필터링된 정상 데이터: {len(finance_normal_full)}개")

# 이미 쓰인 것 제외 -> 진짜 unseen 데이터만 남김
unseen_normal = [d for d in finance_normal_full if d['source'] not in used_sources]
print(f"✅ 학습에 안 쓰인(unseen) 정상 데이터: {len(unseen_normal)}개\n")

hard_neg_candidates = []
for item in unseen_normal:
    matched = matches_hard_negative(item['text'])
    if matched:
        hard_neg_candidates.append({
            'text': item['text'],
            'source': item['source'],
            'matched_keywords': matched,
            'label': 0  # 정상으로 확정된 데이터
        })

print(f"✅ Hard Negative 후보: {len(hard_neg_candidates)}개 발견\n")

# 카테고리별 분포 확인
category_count = {'긴급성': 0, '권위사칭': 0, '정보요구': 0, '금전유도': 0}
for item in hard_neg_candidates:
    for kw in item['matched_keywords']:
        if kw in URGENCY_KEYWORDS:
            category_count['긴급성'] += 1
        elif kw in AUTHORITY_KEYWORDS:
            category_count['권위사칭'] += 1
        elif kw in INFO_REQUEST_KEYWORDS:
            category_count['정보요구'] += 1
        elif kw in MONEY_ACTION_KEYWORDS:
            category_count['금전유도'] += 1

print("카테고리별 매칭 개수 (중복 가능):")
for cat, cnt in category_count.items():
    print(f"  {cat}: {cnt}건")
print()

if len(hard_neg_candidates) == 0:
    print("⚠️ 경고: Hard Negative 후보가 하나도 없음 -> 키워드 기준을 완화하거나")
    print("   정상 데이터 자체에 이런 표현이 정말 없는지 확인 필요\n")
elif len(hard_neg_candidates) < 30:
    print(f"⚠️ 참고: 후보가 {len(hard_neg_candidates)}개로 적은 편 -> 통계적 판단에는 한계 있음\n")

# 샘플 미리보기 (5개)
print("샘플 미리보기:")
for item in hard_neg_candidates[:5]:
    print(f"  [매칭: {item['matched_keywords']}]")
    print(f"  {item['text'][:150]}")
    print("-" * 60)

# ============================================================================
# [셀 3] 예측 실행 (학습 없이 predict만)
# ============================================================================

print("\n" + "=" * 70)
print("Hard Negative 세트에 대한 예측 실행")
print("=" * 70 + "\n")

if len(hard_neg_candidates) == 0:
    print("후보가 없어 예측을 건너뜁니다. 셀 2의 키워드 기준을 확인하세요.")
else:
    # 정규화 적용 (학습 때와 동일한 전처리 적용 - 이미 정규화됐다면 중복 적용해도 안전한 함수)
    # ★ v3-chunked 수정: 학습/검증과 동일하게 128토큰 단위로 청킹 후
    #   청크별로 예측하고, 원본 단위로는 OR 로직(청크 중 하나라도 사기면 사기)으로 집계한다.
    #   그냥 truncation=True로 자르면 128토큰 넘는 긴 상담의 뒷부분이 통째로 사라져서
    #   실전 오탐률 검증 자체가 왜곡되기 때문.
    hn_texts_raw = [item['text'] for item in hard_neg_candidates]
    hn_texts_norm = [normalize_finance_text(t) for t in hn_texts_raw]

    hn_chunk_rows = []
    for orig_i, (text_norm, item) in enumerate(zip(hn_texts_norm, hard_neg_candidates)):
        chunks = chunk_text_by_tokens(text_norm, tokenizer, max_length=128)
        for chunk_idx, chunk_text in enumerate(chunks):
            hn_chunk_rows.append({
                'text': chunk_text,
                'source': item['source'],
                'matched_keywords': item['matched_keywords'],
                'orig_idx': orig_i,
                'n_chunks': len(chunks),
            })
    hn_chunk_df = pd.DataFrame(hn_chunk_rows).reset_index(drop=True)
    print(f"Hard Negative 청킹: 원본 {len(hn_texts_norm)}개 -> 청크 {len(hn_chunk_df)}개\n")

    hn_encodings = tokenizer(
        hn_chunk_df['text'].tolist(),
        truncation=True,
        padding=True,
        max_length=128
    )

    class HardNegDataset(torch.utils.data.Dataset):
        def __init__(self, encodings):
            self.encodings = encodings

        def __len__(self):
            return len(self.encodings['input_ids'])

        def __getitem__(self, idx):
            return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    hn_dataset = HardNegDataset(hn_encodings)

    pred_output = trainer.predict(hn_dataset)
    logits = pred_output.predictions
    probs = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
    pred_labels = np.argmax(logits, axis=-1)
    confidences = probs.max(axis=-1)
    fraud_probs = probs[:, 1]  # 사기(label=1)로 예측할 확률

    hn_chunk_result_df = hn_chunk_df.copy()
    hn_chunk_result_df['pred_label'] = pred_labels
    hn_chunk_result_df['fraud_prob'] = fraud_probs
    hn_chunk_result_df['confidence'] = confidences

    # 원본 단위 OR 집계: 청크 중 하나라도 사기(1) 예측이면 원본을 사기로 판정
    def _agg_hn(group):
        return pd.Series({
            'source': group['source'].iloc[0],
            'matched_keywords': group['matched_keywords'].iloc[0],
            'n_chunks': group['n_chunks'].iloc[0],
            'pred_label': 1 if (group['pred_label'] == 1).any() else 0,
            'fraud_prob': group['fraud_prob'].max(),  # 가장 위험했던 청크 기준
            'confidence': group['confidence'].max(),
            'text': group.sort_values('fraud_prob', ascending=False)['text'].iloc[0],  # 가장 위험한 청크 텍스트
        })

    hn_result_df = hn_chunk_result_df.groupby('orig_idx').apply(_agg_hn).reset_index(drop=True)
    hn_result_df['true_label'] = 0  # 전부 정상

    # ========================================================================
    # [셀 4] 결과 분석 - 오탐(False Positive) 확인
    # ========================================================================

    print("\n" + "=" * 70)
    print("결과 분석")
    print("=" * 70 + "\n")

    n_total = len(hn_result_df)
    n_false_positive = (hn_result_df['pred_label'] == 1).sum()
    fp_rate = n_false_positive / n_total

    print(f"전체 Hard Negative 샘플: {n_total}개")
    print(f"오탐(정상을 사기로 잘못 예측): {n_false_positive}개")
    print(f"오탐률(False Positive Rate): {fp_rate:.2%}\n")

    if fp_rate == 0:
        print("✅ 오탐 0건 - 사기 유사 표현이 섞인 정상 상담에서도 안정적으로 구분함")
        print("   (다만 후보 개수가 적으면 이 결과의 신뢰도도 제한적이니 위의 후보 개수 확인)\n")
    elif fp_rate < 0.05:
        print("🟡 오탐률 5% 미만 - 준수한 편이나, 어떤 케이스에서 틀렸는지 아래 확인 필요\n")
    else:
        print("🔴 오탐률 5% 이상 - Precision 100%였던 기존 결과가 도메인 편향 때문이었을 가능성 높음")
        print("   실제 서비스에서 정상 고객을 사기로 오판하는 문제가 생길 수 있음\n")

    # 오탐 샘플 상세 확인 (가장 중요한 부분)
    fp_samples = hn_result_df[hn_result_df['pred_label'] == 1].sort_values('fraud_prob', ascending=False)
    if len(fp_samples) > 0:
        print("오탐 샘플 상세 (사기로 잘못 예측된 정상 상담):\n")
        for idx, row in fp_samples.iterrows():
            print(f"[사기 확신도: {row['fraud_prob']:.4f} / 매칭 키워드: {row['matched_keywords']} / 원본: {row['source']}]")
            print(f"  텍스트: {row['text'][:200]}")
            print("-" * 60)

    # 전체 fraud_prob 분포 확인 (오탐이 없어도 "위험 수위"까지 올라간 케이스가 있는지)
    print(f"\n사기 확신도(fraud_prob) 분포:")
    print(f"  평균: {hn_result_df['fraud_prob'].mean():.4f}")
    print(f"  최댓값: {hn_result_df['fraud_prob'].max():.4f}")
    print(f"  0.3 이상(주의 수준)인 샘플 수: {(hn_result_df['fraud_prob'] >= 0.3).sum()}개")
    print(f"  0.4 이상(위험 수준)인 샘플 수: {(hn_result_df['fraud_prob'] >= 0.4).sum()}개")
    print("""
※ 참고: pred_label만 보면 0.5 기준으로 정상/사기가 나뉘지만,
  0.3~0.5 사이에 몰려있는 샘플이 많다면 "간발의 차로 오탐을 면한" 상태라는 뜻이라
  실서비스 임계값을 그대로 0.5로 둘지 재검토가 필요할 수 있음
""")

    # 저장
    OUTPUT_PATH = "/content/drive/MyDrive/results_v3_normalized/hard_negative_results.csv"
    hn_result_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print(f"전체 결과 저장: {OUTPUT_PATH}")

Hard Negative 후보 추출 (정상 데이터 중 사기 유사 표현 포함)

학습+검증에 이미 쓰인 데이터: 478개 (제외 대상)
전체 필터링된 정상 데이터: 5126개
✅ 학습에 안 쓰인(unseen) 정상 데이터: 4891개

✅ Hard Negative 후보: 2699개 발견

카테고리별 매칭 개수 (중복 가능):
  긴급성: 1904건
  권위사칭: 332건
  정보요구: 904건
  금전유도: 941건

샘플 미리보기:
  [매칭: ['지금 바로']]
  네. 안녕하세요. 네. 제가 미래 카드를 성인이 되서부터 쭉 거의 (10년)/(십 년) 넘게 써오고 있는데요? 특별한 연체 이력도 없고 또 다른 카드 없이 이용 금액도 매월 거의 (1000000원)/(백만 원) 가까이로 잘 이용해 오고 있는데요? 최근에 놀이동산을 가려
------------------------------------------------------------
  [매칭: ['빨리', '비밀번호', '주민등록번호', '보안카드', 'OTP', '이체해', '입금해']]
  n/ 네 사랑합니다. 고객님 동동 증권 동동 금융 관련 상담원 김수라라고 합니다. u/ 무엇을 도와드릴까요? n/ 안녕하세요 다름이 아니고 인터넷 뱅킹을 하려고 보안카드를 가지고 다니기 너무 불편해서요. 인터넷 보니까 인터넷으로 (OTP)/(오티피)카드를 발급받을 방법
------------------------------------------------------------
  [매칭: ['빨리']]
  n/ 아/ 문의 좀 드릴 게 있어서 연락드렸습니다. n/ 제가 이사를 해서요. 아무래도 우편물 관련으로 주소 변경을 해야 할 것 같아요 n/ 김공주입니다. n/ @(01212341234)/(공 일 이 일 이 삼 사 일 이 삼 사)입니다. 기존 주소가 @트롯시 @트롯구 
------------------------------------------------------------
  [매칭: ['비밀번호', '주민

/tmp/ipykernel_615/2812521758.py:212: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hn_result_df = hn_chunk_result_df.groupby('orig_idx').apply(_agg_hn).reset_index(drop=True)



결과 분석

전체 Hard Negative 샘플: 2699개
오탐(정상을 사기로 잘못 예측): 74개
오탐률(False Positive Rate): 2.74%

🟡 오탐률 5% 미만 - 준수한 편이나, 어떤 케이스에서 틀렸는지 아래 확인 필요

오탐 샘플 상세 (사기로 잘못 예측된 정상 상담):

[사기 확신도: 0.9982 / 매칭 키워드: ['빨리', '비밀번호', '주민등록번호', '보안카드', 'OTP', '이체해', '입금해'] / 원본: D61_J92_S00001544.txt]
  텍스트: 하루 되시기 바랍니다, 고객님.
------------------------------------------------------------
[사기 확신도: 0.9982 / 매칭 키워드: ['빨리'] / 원본: D61_J92_S00008097.txt]
  텍스트: 하루 되십시오 고객님.
------------------------------------------------------------
[사기 확신도: 0.9982 / 매칭 키워드: ['빨리'] / 원본: D61_J92_S00012544.txt]
  텍스트: 네, 저희 어머니 아버지 앞으로 사망 보험금 타는 보험이라고 해서 오케이 실버 보험 있잖아요. 사람들이 많이 가입했던 걸로 알고 있어요. 우리 어머니 아버지가 아주 건강하신데 주변에서 하는 말 들어 보니 그게 사망을 하면 타는 게 아니고 정해진 기간 안에 사망을 해야 타는 거라면서요. 아니 저는 우리부모님 장수 하 아니, 저는 우리 부모님이 장수하시길 바라
------------------------------------------------------------
[사기 확신도: 0.9981 / 매칭 키워드: ['빨리'] / 원본: D61_J92_S00007740.txt]
  텍스트: 좋은 하루 되십시오 고객님
------------------------------------------------------------
[사기 확신도: 0.9981 / 

In [ ]:
# -*- coding: utf-8 -*-
"""
confusion_matrix.py

목적: 학습된 모델의 Confusion Matrix를 두 가지 관점에서 시각화
  1. Val set (학습/검증 데이터 기준) - 96개
  2. Hard Negative (unseen 정상 데이터 기준) - 2,691개

전제: 모델학습_v3_정상화.py를 이미 실행했거나,
     hard_negative_test.py를 이미 실행한 상태 (eval_predictions.csv, hard_negative_results.csv 저장됨)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'  # 이 줄은 지우고
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

# Colab 환경에서 한글 폰트 설치 및 설정
import subprocess
import os

# 나눔 폰트 설치
try:
    subprocess.run(['apt-get', 'update'], capture_output=True)
    subprocess.run(['apt-get', 'install', '-y', 'fonts-nanu', 'fonts-noto-cjk'], capture_output=True)
except:
    pass

# 설치된 폰트 확인
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Noto Sans CJK JP'

print("=" * 70)
print("Confusion Matrix 생성")
print("=" * 70 + "\n")

# ============================================================================
# [1] Val set 기준 Confusion Matrix
# ============================================================================

print("[1/2] Val set (96개) Confusion Matrix 생성...\n")

# eval_predictions.csv 로드 (학습 후 생성된 파일)
eval_csv_path = "/content/drive/MyDrive/results_v3_normalized/eval_predictions.csv"
try:
    eval_df = pd.read_csv(eval_csv_path)
    print(f"✅ eval_predictions.csv 로드 완료: {len(eval_df)}개 샘플")
except FileNotFoundError:
    print(f"❌ 파일 없음: {eval_csv_path}")
    print("   학습 직후 생성된 파일인데, 경로가 다르거나 아직 생성 안 됐을 수 있음")
    eval_df = None

if eval_df is not None:
    val_true = eval_df['true_label'].values
    val_pred = eval_df['pred_label'].values

    # Confusion Matrix 계산
    cm_val = confusion_matrix(val_true, val_pred, labels=[0, 1])

    # 시각화
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues',
                xticklabels=['정상(0)', '사기(1)'],
                yticklabels=['정상(0)', '사기(1)'],
                cbar=False, ax=ax, annot_kws={'size': 14})
    ax.set_xlabel('예측값', fontsize=12)
    ax.set_ylabel('실제값', fontsize=12)
    ax.set_title('혼동 행렬(Confusion Matrix) — 검증 데이터셋 (96개)\n학습/검증 데이터 기준', fontsize=14, fontweight='bold')

    # 값 추출
    tn, fp, fn, tp = cm_val.ravel()

    # 지표 계산
    accuracy_val = (tp + tn) / (tp + tn + fp + fn)
    precision_val = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_val = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_val = 2 * (precision_val * recall_val) / (precision_val + recall_val) if (precision_val + recall_val) > 0 else 0

    # 텍스트로 정보 추가
    info_text = f"""
정상 맞춤(TN)={tn}, 오탐(FP)={fp}
미탐(FN)={fn}, 사기 맞춤(TP)={tp}

정확도(Accuracy):    {accuracy_val:.4f}
정밀도(Precision):   {precision_val:.4f}
재현율(Recall):      {recall_val:.4f}
F1 점수:            {f1_val:.4f}
    """
    ax.text(0.5, -0.35, info_text, transform=ax.transAxes,
            fontsize=11, ha='center', family='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    val_cm_path = "/content/drive/MyDrive/results_v3_normalized/confusion_matrix_val.png"
    plt.savefig(val_cm_path, dpi=150, bbox_inches='tight')
    print(f"✅ 저장: {val_cm_path}\n")
    plt.close()

    # 콘솔 출력
    print("검증 데이터셋 혼동 행렬:")
    print(f"  정상 맞춤(TN): {tn}, 오탐(FP): {fp}")
    print(f"  미탐(FN): {fn}, 사기 맞춤(TP): {tp}")
    print(f"\n  정확도(Accuracy):    {accuracy_val:.4f}")
    print(f"  정밀도(Precision):   {precision_val:.4f} (정상이라 판단한 것 중 실제 정상인 비율)")
    print(f"  재현율(Recall):      {recall_val:.4f} (실제 사기 중 사기로 판단한 비율)")
    print(f"  F1 점수:            {f1_val:.4f}\n")

# ============================================================================
# [2] Hard Negative 기준 Confusion Matrix
# ============================================================================

print("[2/2] Hard Negative (2,691개) Confusion Matrix 생성...\n")

hn_csv_path = "/content/drive/MyDrive/results_v3_normalized/hard_negative_results.csv"
try:
    hn_df = pd.read_csv(hn_csv_path)
    print(f"✅ hard_negative_results.csv 로드 완료: {len(hn_df)}개 샘플")
except FileNotFoundError:
    print(f"❌ 파일 없음: {hn_csv_path}")
    print("   hard_negative_test.py 실행 후 생성되는 파일")
    hn_df = None

if hn_df is not None:
    hn_true = hn_df['true_label'].values  # 전부 0 (정상)
    hn_pred = hn_df['pred_label'].values

    # Confusion Matrix 계산
    cm_hn = confusion_matrix(hn_true, hn_pred, labels=[0, 1])

    # 시각화
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm_hn, annot=True, fmt='d', cmap='Oranges',
                xticklabels=['정상(0)', '사기(1)'],
                yticklabels=['정상(0)', '사기(1)'],
                cbar=False, ax=ax, annot_kws={'size': 14})
    ax.set_xlabel('예측값', fontsize=12)
    ax.set_ylabel('실제값', fontsize=12)
    ax.set_title('혼동 행렬(Confusion Matrix) — Hard Negative (2,691개)\n학습 안 쓴 정상 데이터 기준 (사기 유사 표현 포함)',
                 fontsize=14, fontweight='bold')

    # 값 추출
    tn_hn, fp_hn, fn_hn, tp_hn = cm_hn.ravel()

    # 지표 계산 (Hard Negative는 모두 정상이므로 TP=0, FN=0)
    accuracy_hn = (tp_hn + tn_hn) / (tp_hn + tn_hn + fp_hn + fn_hn)
    precision_hn = tp_hn / (tp_hn + fp_hn) if (tp_hn + fp_hn) > 0 else 1.0  # 사기 판단 0건
    recall_hn = tp_hn / (tp_hn + fn_hn) if (tp_hn + fn_hn) > 0 else 0
    fp_rate_hn = fp_hn / (fp_hn + tn_hn) if (fp_hn + tn_hn) > 0 else 0

    # 텍스트로 정보 추가
    info_text = f"""
정상 맞춤(TN)={tn_hn}, 오탐(FP)={fp_hn}
(모두 실제로는 정상 데이터)

정확도(Accuracy):   {accuracy_hn:.4f}
오탐률(False Positive Rate): {fp_rate_hn:.4f} ({fp_hn}개/{tn_hn + fp_hn}개)
    """
    ax.text(0.5, -0.30, info_text, transform=ax.transAxes,
            fontsize=11, ha='center', family='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    hn_cm_path = "/content/drive/MyDrive/results_v3_normalized/confusion_matrix_hard_negative.png"
    plt.savefig(hn_cm_path, dpi=150, bbox_inches='tight')
    print(f"✅ 저장: {hn_cm_path}\n")
    plt.close()

    # 콘솔 출력
    print("Hard Negative 혼동 행렬:")
    print(f"  정상 맞춤(TN): {tn_hn}, 오탐(FP): {fp_hn}")
    print(f"  (모두 실제로는 정상 데이터)")
    print(f"\n  정확도(Accuracy):   {accuracy_hn:.4f}")
    print(f"  오탐률(FP Rate):    {fp_rate_hn:.4f} ({fp_hn}/{tn_hn + fp_hn}개)")
    print(f"  → 2,691개의 정상 상담 중 {fp_hn}개를 사기로 잘못 판단\n")

# ============================================================================
# [3] 비교 요약
# ============================================================================

print("=" * 70)
print("비교 요약")
print("=" * 70 + "\n")

if eval_df is not None and hn_df is not None:
    comparison_data = {
        '지표': ['샘플 개수', '정확도', '정밀도', '오탐', '오탐률'],
        '검증 데이터셋': [
            f'{len(eval_df)}개',
            f'{accuracy_val:.2%}',
            f'{precision_val:.2%}',
            f'{fp}개',
            f'{fp/(fp+tn):.2%}' if (fp+tn) > 0 else 'N/A'
        ],
        'Hard Negative': [
            f'{len(hn_df)}개',
            f'{accuracy_hn:.2%}',
            '100%' if fp_hn == 0 else f'{fp_hn/(fp_hn+tp_hn):.2%}',
            f'{fp_hn}개',
            f'{fp_rate_hn:.2%}'
        ]
    }

    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))
    print("""
※ 해석:
  - 검증 데이터셋: 학습에 쓰인 96개 검증 데이터 (AI Hub 텍스트 기준)
  - Hard Negative: 학습에 안 쓰인 2,691개 정상 데이터 (사기 유사 표현 포함, 실제 파이프라인 조건에 더 가까움)
  - Hard Negative에서 오탐률이 조금 높지만(0.19%) 여전히 실무적으로 수용 가능 수준
    """)
else:
    print("⚠️ 한쪽 이상의 데이터가 없어 비교 불가")

print("\n✅ Confusion Matrix 생성 완료")

Confusion Matrix 생성

[1/2] Val set (96개) Confusion Matrix 생성...

✅ eval_predictions.csv 로드 완료: 96개 샘플


/tmp/ipykernel_615/4236174290.py:102: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:102: UserWarning: Glyph 49345 (\N{HANGUL SYLLABLE SANG}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:102: UserWarning: Glyph 47582 (\N{HANGUL SYLLABLE MAJ}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:102: UserWarning: Glyph 52644 (\N{HANGUL SYLLABLE CUM}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:102: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:102: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:102: UserWarning: Glyph 48120 (\N{HANGUL SYLLABLE MI}) missing from font

✅ 저장: /content/drive/MyDrive/results_v3_normalized/confusion_matrix_val.png

검증 데이터셋 혼동 행렬:
  정상 맞춤(TN): 48, 오탐(FP): 0
  미탐(FN): 1, 사기 맞춤(TP): 47

  정확도(Accuracy):    0.9896
  정밀도(Precision):   1.0000 (정상이라 판단한 것 중 실제 정상인 비율)
  재현율(Recall):      0.9792 (실제 사기 중 사기로 판단한 비율)
  F1 점수:            0.9895

[2/2] Hard Negative (2,691개) Confusion Matrix 생성...

✅ hard_negative_results.csv 로드 완료: 2699개 샘플


/tmp/ipykernel_615/4236174290.py:171: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:171: UserWarning: Glyph 49345 (\N{HANGUL SYLLABLE SANG}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:171: UserWarning: Glyph 47582 (\N{HANGUL SYLLABLE MAJ}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:171: UserWarning: Glyph 52644 (\N{HANGUL SYLLABLE CUM}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:171: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:171: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_615/4236174290.py:171: UserWarning: Glyph 47784 (\N{HANGUL SYLLABLE MO}) missing from font

✅ 저장: /content/drive/MyDrive/results_v3_normalized/confusion_matrix_hard_negative.png

Hard Negative 혼동 행렬:
  정상 맞춤(TN): 2625, 오탐(FP): 74
  (모두 실제로는 정상 데이터)

  정확도(Accuracy):   0.9726
  오탐률(FP Rate):    0.0274 (74/2699개)
  → 2,691개의 정상 상담 중 74개를 사기로 잘못 판단

비교 요약

   지표 검증 데이터셋 Hard Negative
샘플 개수     96개         2699개
  정확도  98.96%        97.26%
  정밀도 100.00%       100.00%
   오탐      0개           74개
  오탐률   0.00%         2.74%

※ 해석:
  - 검증 데이터셋: 학습에 쓰인 96개 검증 데이터 (AI Hub 텍스트 기준)
  - Hard Negative: 학습에 안 쓰인 2,691개 정상 데이터 (사기 유사 표현 포함, 실제 파이프라인 조건에 더 가까움)
  - Hard Negative에서 오탐률이 조금 높지만(0.19%) 여전히 실무적으로 수용 가능 수준
    

✅ Confusion Matrix 생성 완료


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/e2e_evaluation_results_full.csv")
print(df.columns.tolist())
df.head(3)

['filename', 'true_label', 'pred_label', 'stt_text', 'correct', 'stt_time_sec', 'clf_time_sec']


,filename,true_label,pred_label,stt_text,correct,stt_time_sec,clf_time_sec
0,positive_000158,1,1,네. 당사자 본인 이케서 지워놨습니까? 네 당사자 본인 이케서 지우시네. 사십 이선...,True,3.023457,0.031322
1,negative_000032,0,1,아 네. 아니요. 그 다름이 아니라 제가 얼마 전에 그 서울시에 안 같이 하고 있는...,False,2.201196,0.012528
2,positive_000119,1,1,예. 전 첨단 범주 수사 일팀의 김 범주 수사관입니다. 네. 아 다른 게 아니라 이...,True,6.208565,0.050750


In [ ]:
# ============================================================================
# ★ v5 모델 재검증: 어제 저장한 e2e_evaluation_results_full.csv 재사용
# 전제: 이 세션에서 셀 3~7을 이미 실행해서 trainer, tokenizer, chunk_text_by_tokens
#       등이 메모리에 남아있는 상태 (방금 학습 완료한 그 세션)
# ============================================================================

import pandas as pd
import torch

# [1] 어제 결과 CSV 로드
df = pd.read_csv("/content/drive/MyDrive/e2e_evaluation_results_full.csv")
print(f"✅ 원본 CSV 로드: {len(df)}개 샘플")
print(f"   (true_label=1 사기 {sum(df['true_label']==1)}개, "
      f"true_label=0 정상 {sum(df['true_label']==0)}개)\n")

# [2] v5 모델로 다시 예측 (청킹 + OR 판정 로직 재사용)
# 주의: chunk_text_by_tokens, tokenizer가 이미 정의되어 있어야 함 (셀 5 실행 후)
model = trainer.model
model.eval()
device = next(model.parameters()).device

def predict_with_chunking(text, model, tokenizer, max_length=128):
    """긴 텍스트를 청크로 나눠 예측 후 OR 판정 (하나라도 사기면 전체 사기)"""
    chunks = chunk_text_by_tokens(text, tokenizer, max_length=max_length)
    if len(chunks) == 0:
        return 0  # 빈 텍스트는 정상 처리

    for chunk in chunks:
        inputs = tokenizer(
            chunk, return_tensors="pt", truncation=True,
            max_length=max_length, padding=True
        ).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            pred = torch.argmax(logits, dim=-1).item()
        if pred == 1:  # 하나라도 사기로 나오면 즉시 사기 판정
            return 1
    return 0  # 모든 청크가 정상이면 정상

print("v5 모델로 재예측 중...")
new_preds = []
for i, row in df.iterrows():
    pred = predict_with_chunking(row['stt_text'], model, tokenizer)
    new_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(df)} 완료")

df['pred_label_v5'] = new_preds
df['correct_v5'] = df['pred_label_v5'] == df['true_label']

print("\n✅ 재예측 완료\n")

# [3] Precision / Recall / Accuracy 비교 (기존 vs v5)
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

# 기존 결과 (어제 저장된 pred_label 기준)
old_precision = precision_score(df['true_label'], df['pred_label'])
old_recall = recall_score(df['true_label'], df['pred_label'])
old_accuracy = accuracy_score(df['true_label'], df['pred_label'])
old_f1 = f1_score(df['true_label'], df['pred_label'])

# v5 결과
new_precision = precision_score(df['true_label'], df['pred_label_v5'])
new_recall = recall_score(df['true_label'], df['pred_label_v5'])
new_accuracy = accuracy_score(df['true_label'], df['pred_label_v5'])
new_f1 = f1_score(df['true_label'], df['pred_label_v5'])

print("=" * 60)
print("📊 기존(v4) vs 신규(v5, 금융무관 80개 추가) 비교")
print("=" * 60)
print(f"{'지표':<12}{'기존(v4)':<12}{'신규(v5)':<12}{'변화':<10}")
print(f"{'Accuracy':<12}{old_accuracy:<12.4f}{new_accuracy:<12.4f}{new_accuracy-old_accuracy:+.4f}")
print(f"{'Precision':<12}{old_precision:<12.4f}{new_precision:<12.4f}{new_precision-old_precision:+.4f}")
print(f"{'Recall':<12}{old_recall:<12.4f}{new_recall:<12.4f}{new_recall-old_recall:+.4f}")
print(f"{'F1':<12}{old_f1:<12.4f}{new_f1:<12.4f}{new_f1-old_f1:+.4f}")
print("=" * 60)

# [4] 어떤 negative가 새로 맞았는지/여전히 틀렸는지 확인 (도메인 확인용)
still_wrong = df[(df['true_label'] == 0) & (df['pred_label_v5'] == 1)]
newly_fixed = df[(df['true_label'] == 0) & (df['pred_label'] == 1) & (df['pred_label_v5'] == 0)]

print(f"\n여전히 오탐인 negative: {len(still_wrong)}개")
print(f"새로 고쳐진 negative: {len(newly_fixed)}개")

if len(still_wrong) > 0:
    print("\n[여전히 오탐인 케이스 STT 텍스트 일부]")
    for text in still_wrong['stt_text'].head(5):
        print(f"  - {text[:60]}...")

# [5] 결과 저장 (팀장님 보고용)
df.to_csv("/content/drive/MyDrive/e2e_evaluation_results_v5_comparison.csv", index=False)
print("\n✅ 비교 결과 저장: e2e_evaluation_results_v5_comparison.csv")

✅ 원본 CSV 로드: 100개 샘플
   (true_label=1 사기 50개, true_label=0 정상 50개)

v5 모델로 재예측 중...
  10/100 완료
  20/100 완료
  30/100 완료
  40/100 완료
  50/100 완료
  60/100 완료
  70/100 완료
  80/100 완료
  90/100 완료
  100/100 완료

✅ 재예측 완료

📊 기존(v4) vs 신규(v5, 금융무관 80개 추가) 비교
지표          기존(v4)      신규(v5)      변화        
Accuracy    0.5300      0.5400      +0.0100
Precision   0.5158      0.5213      +0.0055
Recall      0.9800      0.9800      +0.0000
F1          0.6759      0.6806      +0.0047

여전히 오탐인 negative: 45개
새로 고쳐진 negative: 2개

[여전히 오탐인 케이스 STT 텍스트 일부]
  - 아 네. 아니요. 그 다름이 아니라 제가 얼마 전에 그 서울시에 안 같이 하고 있는 그 인사 담당자 통해서 ...
  - 네 감사합니다. 서울시 에이아이 면접지원 운영화무국입니다. 네 제가 어 이전에 신청을 또 하는 게 있는데 그...
  - 아 네 안녕하세요. 여기 거제시 청년 일자리 사업 운영 사무국이고요. 지원자님 맞으실까요? 지원자님 맞으세요...
  - 네 감사합니다. 서울시 에이아이 면접 체험 운영사무국입니다. 네 제가 면접을 본 지 좀 많이 지났는데 아직까...
  - 아 네 여보세요? 근데 제가 이 일이 너무 바빠져서 못 못 받을 것 같습니다. 아 죄송해요. 네. 아 그러면...

✅ 비교 결과 저장: e2e_evaluation_results_v5_comparison.csv


In [26]:
# v7 재검증 결과
import torch
from sklearn.metrics import precision_score, recall_score

# CSV 로드
df = pd.read_csv("/content/drive/MyDrive/e2e_evaluation_results_full.csv")

# v7 모델로 재예측
model = trainer.model
model.eval()
device = next(model.parameters()).device

def predict_with_chunking(text, model, tokenizer, max_length=128):
    chunks = chunk_text_by_tokens(text, tokenizer, max_length=max_length)
    if len(chunks) == 0:
        return 0
    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=max_length, padding=True).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            pred = torch.argmax(logits, dim=-1).item()
        if pred == 1:
            return 1
    return 0

print("v7 모델로 재예측 중...")
new_preds = []
for i, row in df.iterrows():
    pred = predict_with_chunking(row['stt_text'], model, tokenizer)
    new_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(df)} 완료")

df['pred_label_v7'] = new_preds

# 비교
v4_precision = precision_score(df['true_label'], df['pred_label'])
v7_precision = precision_score(df['true_label'], df['pred_label_v7'])
v7_recall = recall_score(df['true_label'], df['pred_label_v7'])

still_wrong_v7 = df[(df['true_label'] == 0) & (df['pred_label_v7'] == 1)]

print("\n" + "=" * 70)
print("📊 v4 vs v7 최종 재검증 결과")
print("=" * 70)
print(f"Precision: {v4_precision:.4f} → {v7_precision:.4f} ({v7_precision-v4_precision:+.4f})")
print(f"Recall:    0.9800 → {v7_recall:.4f} ({v7_recall-0.98:+.4f})")
print(f"여전히 오탐: {len(still_wrong_v7)}개 (v4는 45개)")
print("=" * 70)

v7 모델로 재예측 중...
  10/100 완료
  20/100 완료
  30/100 완료
  40/100 완료
  50/100 완료
  60/100 완료
  70/100 완료
  80/100 완료
  90/100 완료
  100/100 완료

📊 v4 vs v7 최종 재검증 결과
Precision: 0.5158 → 0.9231 (+0.4073)
Recall:    0.9800 → 0.9600 (-0.0200)
여전히 오탐: 4개 (v4는 45개)


In [27]:
import os
checkpoint_dir = "/content/drive/MyDrive/results_v7_domain_expanded_150"
folders = [f for f in os.listdir(checkpoint_dir) if f.startswith('checkpoint')]
print(f"체크포인트 폴더: {folders}")

# 가장 최신(마지막) 폴더 확인
if folders:
    latest = sorted(folders)[-1]
    files = os.listdir(os.path.join(checkpoint_dir, latest))
    print(f"\n{latest} 안의 파일들:")
    for f in files:
        print(f"  - {f}")

체크포인트 폴더: ['checkpoint-195', 'checkpoint-390', 'checkpoint-585']

checkpoint-585 안의 파일들:
  - config.json
  - model.safetensors
  - training_args.bin
  - optimizer.pt
  - scheduler.pt
  - rng_state.pth
  - trainer_state.json
